# 03 — Model Comparison and Final Evaluation

**Project:** Bangladeshi Freshwater Fish Image Classification Using CNN and Transfer Learning
**Dataset:** BFF-15 (15 classes)
**Owner:** Swapnil Santosh Akhade

## Purpose

Read the logs and metrics produced by:

- `01_custom_cnn.ipynb` (Vivek)
- `02_transfer_learning.ipynb` (Mansi)

and produce:

1. Side-by-side training curves (accuracy and loss).
2. A comparison table of test-set metrics.
3. Confusion matrices side by side.
4. A short written comparison, kept separate from the published paper's results.

## What this notebook needs

- `custom_cnn/logs/training_log.csv`
- `custom_cnn/metrics/final_metrics.json`
- `custom_cnn/metrics/confusion_matrix.csv`
- `custom_cnn/plots/confusion_matrix.png`
- `transfer_learning/logs/training_log.csv`
- `transfer_learning/metrics/final_metrics.json`
- `transfer_learning/metrics/confusion_matrix.csv`
- `transfer_learning/plots/confusion_matrix.png`

If any are missing, both teammates haven't finished yet.

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CNN_DIR = PROJECT_ROOT / "custom_cnn"
TL_DIR  = PROJECT_ROOT / "transfer_learning"
OUT_DIR = PROJECT_ROOT / "outputs" / "comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

required = [
    CNN_DIR / "logs" / "training_log.csv",
    CNN_DIR / "metrics" / "final_metrics.json",
    CNN_DIR / "metrics" / "confusion_matrix.csv",
    TL_DIR  / "logs" / "training_log.csv",
    TL_DIR  / "metrics" / "final_metrics.json",
    TL_DIR  / "metrics" / "confusion_matrix.csv",
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    print("Missing artifacts:")
    for m in missing:
        print(" -", m)
    raise SystemExit("Both teammates must finish training first.")

print("All artifacts present.")

In [ ]:
cnn_log = pd.read_csv(CNN_DIR / "logs" / "training_log.csv")
tl_log  = pd.read_csv(TL_DIR  / "logs" / "training_log.csv")

with open(CNN_DIR / "metrics" / "final_metrics.json") as f:
    cnn_metrics = json.load(f)
with open(TL_DIR / "metrics" / "final_metrics.json") as f:
    tl_metrics = json.load(f)

cnn_cm = pd.read_csv(CNN_DIR / "metrics" / "confusion_matrix.csv", index_col=0)
tl_cm  = pd.read_csv(TL_DIR  / "metrics" / "confusion_matrix.csv", index_col=0)

print("CNN epochs:", len(cnn_log))
print("TL  epochs:", len(tl_log))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(cnn_log["epoch"], cnn_log["accuracy"], label="train")
axes[0, 0].plot(cnn_log["epoch"], cnn_log["val_accuracy"], label="val")
axes[0, 0].set_title("Custom CNN — Accuracy")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].legend()

axes[0, 1].plot(tl_log["epoch"], tl_log["accuracy"], label="train")
axes[0, 1].plot(tl_log["epoch"], tl_log["val_accuracy"], label="val")
axes[0, 1].set_title(f"{tl_metrics['base_model']} — Accuracy")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].legend()

axes[1, 0].plot(cnn_log["epoch"], cnn_log["loss"], label="train")
axes[1, 0].plot(cnn_log["epoch"], cnn_log["val_loss"], label="val")
axes[1, 0].set_title("Custom CNN — Loss")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].legend()

axes[1, 1].plot(tl_log["epoch"], tl_log["loss"], label="train")
axes[1, 1].plot(tl_log["epoch"], tl_log["val_loss"], label="val")
axes[1, 1].set_title(f"{tl_metrics['base_model']} — Loss")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(str(OUT_DIR / "curves_side_by_side.png"), dpi=150)
plt.show()

In [ ]:
rows = [
    ("Accuracy",            cnn_metrics["test_accuracy"],          tl_metrics["test_accuracy"]),
    ("Precision (macro)",   cnn_metrics["test_precision_macro"],   tl_metrics["test_precision_macro"]),
    ("Recall (macro)",      cnn_metrics["test_recall_macro"],      tl_metrics["test_recall_macro"]),
    ("F1 (macro)",          cnn_metrics["test_f1_macro"],          tl_metrics["test_f1_macro"]),
    ("Precision (weighted)",cnn_metrics["test_precision_weighted"],tl_metrics["test_precision_weighted"]),
    ("Recall (weighted)",   cnn_metrics["test_recall_weighted"],   tl_metrics["test_recall_weighted"]),
    ("F1 (weighted)",       cnn_metrics["test_f1_weighted"],       tl_metrics["test_f1_weighted"]),
    ("Epochs ran",          cnn_metrics["epochs_ran"],             tl_metrics["epochs_ran"]),
]

df = pd.DataFrame(rows, columns=["Metric", "Custom CNN", f"TL ({tl_metrics['base_model']})"])
df["Δ (TL - CNN)"] = df.iloc[:, 2] - df.iloc[:, 1]

print(df.to_string(index=False))
df.to_csv(OUT_DIR / "comparison_table.csv", index=False)
print()
print("Saved:", OUT_DIR / "comparison_table.csv")

In [ ]:
cm_cnn = cnn_cm.values
cm_tl  = tl_cm.values
labels = list(cnn_cm.index)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, cm, title in [
    (axes[0], cm_cnn, "Custom CNN"),
    (axes[1], cm_tl,  f"Transfer Learning ({tl_metrics['base_model']})"),
]:
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90)
    ax.set_yticklabels(labels)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig(str(OUT_DIR / "confusion_matrices_side_by_side.png"), dpi=150)
plt.show()

## Written comparison

Fill in after looking at the table and the confusion matrices.

Things to discuss:

- Which model performed better on the test set, and by how much.
- Which classes are hardest for both models (rows with many off-diagonal entries).
- Which classes one model handles but the other struggles with.
- Whether the transfer-learning model overfit, underfit, or generalized well
  (look at train vs val curves).
- Whether the custom CNN had enough capacity or training time.
- Whether the class imbalance affected either model.

**Do not** claim we reproduced the reference paper. The paper uses a different
dataset (8 species), different models, and a different augmentation strategy.
Our results are an adapted comparison, not a reproduction.

## Reference paper vs our experiment

| Item | Reference paper | This practical |
|---|---|---|
| Dataset | 8 species | BFF-15, 15 classes |
| Train/test split | Not identical to ours | 70 / 15 / 15 stratified, seed 42 |
| Models | InceptionV3, Xception, DenseNet, InceptionResNetV2, ResNet152V2, hybrid CNN–ConvLSTM | One from-scratch CNN + one fine-tuned pretrained model |
| Reported top accuracy | To be verified from the paper | See table above |

Both rows are deliberately separated. Do not merge them.

In [ ]:
summary = {
    "custom_cnn": {
        "accuracy": cnn_metrics["test_accuracy"],
        "f1_macro": cnn_metrics["test_f1_macro"],
        "f1_weighted": cnn_metrics["test_f1_weighted"],
        "epochs_ran": cnn_metrics["epochs_ran"],
    },
    "transfer_learning": {
        "base_model": tl_metrics["base_model"],
        "accuracy": tl_metrics["test_accuracy"],
        "f1_macro": tl_metrics["test_f1_macro"],
        "f1_weighted": tl_metrics["test_f1_weighted"],
        "epochs_ran": tl_metrics["epochs_ran"],
    },
}

with open(OUT_DIR / "final_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))